In [0]:
# Célula 1 — verificar arquivos no Volume
display(dbutils.fs.ls("/Volumes/workspace/raw/landing/"))

path,name,size,modificationTime
dbfs:/Volumes/workspace/raw/landing/olist_customers_dataset.csv,olist_customers_dataset.csv,9033957,1780598829000
dbfs:/Volumes/workspace/raw/landing/olist_geolocation_dataset.csv,olist_geolocation_dataset.csv,61273883,1780598841000
dbfs:/Volumes/workspace/raw/landing/olist_order_items_dataset.csv,olist_order_items_dataset.csv,15438671,1780598830000
dbfs:/Volumes/workspace/raw/landing/olist_order_payments_dataset.csv,olist_order_payments_dataset.csv,5777138,1780598827000
dbfs:/Volumes/workspace/raw/landing/olist_order_reviews_dataset.csv,olist_order_reviews_dataset.csv,14451670,1780598830000
dbfs:/Volumes/workspace/raw/landing/olist_orders_dataset.csv,olist_orders_dataset.csv,17654914,1780598831000
dbfs:/Volumes/workspace/raw/landing/olist_products_dataset.csv,olist_products_dataset.csv,2379446,1780598823000
dbfs:/Volumes/workspace/raw/landing/olist_sellers_dataset.csv,olist_sellers_dataset.csv,174703,1780598812000
dbfs:/Volumes/workspace/raw/landing/product_category_name_translation.csv,product_category_name_translation.csv,2613,1780598810000


In [0]:
# Célula 2 — ingestão Raw: leitura dos CSVs e salvamento como Delta Tables
from pyspark.sql.functions import current_timestamp, lit

BASE_PATH = "/Volumes/workspace/raw/landing/"
BASE_CATALOG = "workspace.raw"

datasets = [
    "olist_customers_dataset",
    "olist_geolocation_dataset",
    "olist_order_items_dataset",
    "olist_order_payments_dataset",
    "olist_order_reviews_dataset",
    "olist_orders_dataset",
    "olist_products_dataset",
    "olist_sellers_dataset",
    "product_category_name_translation",
]

for ds in datasets:
    print(f"Ingerindo: {ds}")
    df = spark.read.csv(
        f"{BASE_PATH}{ds}.csv",
        header=True,
        inferSchema=True
    )
    df = df \
        .withColumn("_ingested_at", current_timestamp()) \
        .withColumn("_source_file", lit(f"{ds}.csv"))
    
    df.write.format("delta") \
        .mode("overwrite") \
        .saveAsTable(f"{BASE_CATALOG}.{ds}")
    
    print(f" {df.count()} linhas salvas em {BASE_CATALOG}.{ds}")

print("\nIngestão Raw concluída!")

Ingerindo: olist_customers_dataset
  ✓ 99441 linhas salvas em workspace.raw.olist_customers_dataset
Ingerindo: olist_geolocation_dataset
  ✓ 1000163 linhas salvas em workspace.raw.olist_geolocation_dataset
Ingerindo: olist_order_items_dataset
  ✓ 112650 linhas salvas em workspace.raw.olist_order_items_dataset
Ingerindo: olist_order_payments_dataset
  ✓ 103886 linhas salvas em workspace.raw.olist_order_payments_dataset
Ingerindo: olist_order_reviews_dataset
  ✓ 104162 linhas salvas em workspace.raw.olist_order_reviews_dataset
Ingerindo: olist_orders_dataset
  ✓ 99441 linhas salvas em workspace.raw.olist_orders_dataset
Ingerindo: olist_products_dataset
  ✓ 32951 linhas salvas em workspace.raw.olist_products_dataset
Ingerindo: olist_sellers_dataset
  ✓ 3095 linhas salvas em workspace.raw.olist_sellers_dataset
Ingerindo: product_category_name_translation
  ✓ 71 linhas salvas em workspace.raw.product_category_name_translation

Ingestão Raw concluída!


In [0]:
# Célula 3 — comentários de coluna no Unity Catalog
spark.sql("""
  ALTER TABLE workspace.raw.olist_orders_dataset
  ALTER COLUMN order_id COMMENT 'Identificador único do pedido'
""")
spark.sql("""
  ALTER TABLE workspace.raw.olist_orders_dataset
  ALTER COLUMN customer_id COMMENT 'Chave estrangeira para a tabela de clientes'
""")
spark.sql("""
  ALTER TABLE workspace.raw.olist_orders_dataset
  ALTER COLUMN order_status COMMENT 'Status do pedido: delivered, shipped, canceled, etc'
""")
spark.sql("""
  ALTER TABLE workspace.raw.olist_customers_dataset
  ALTER COLUMN customer_state COMMENT 'UF do cliente'
""")
spark.sql("""
  ALTER TABLE workspace.raw.olist_order_payments_dataset
  ALTER COLUMN payment_value COMMENT 'Valor pago em reais'
""")

print("Comentários registrados no Unity Catalog!")

Comentários registrados no Unity Catalog!
